In [23]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PROJECT PATHS
# =========================================================

# =========================================================
# PROJECT ROOT
# =========================================================

import os
from pathlib import Path


def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)

PROCESSED = PROJECT_ROOT / "data" / "processed"
FEATURES = PROJECT_ROOT / "data" / "features"
RESULTS = PROJECT_ROOT / "results"

RESULTS.mkdir(parents=True, exist_ok=True)
FEATURES.mkdir(parents=True, exist_ok=True)

CUTOFF = "2011-01-01"

# =========================================================
# INPUT / OUTPUT FILES
# =========================================================

EMAIL_DAILY = PROCESSED / "email_daily_r62.parquet"
USB_DAILY = PROCESSED / "usb_daily_r62.parquet"
HTTP_DAILY = PROCESSED / "http_daily_r62.parquet"
LOGON_DAILY = PROCESSED / "logon_daily_r62.parquet"

USER_DAY = PROCESSED / "user_day_r62_v1.parquet"

GT_USERS = PROCESSED / "ground_truth_users_r62.parquet"
GT_EVENTS = PROCESSED / "ground_truth_events_r62.parquet"

BASELINE = FEATURES / "user_baseline_r62_v1.parquet"
BEHAVIOR = FEATURES / "behavior_features_r62_v1.parquet"

MALICIOUS_BASELINE = (
    FEATURES /
    "malicious_pre_attack_baseline_r62.parquet"
)

ATTACK_HOSTS = (
    FEATURES /
    "attack_host_eligibility_r62.parquet"
)

print("✅ SUCCESS — Final QA setup ready.")

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
✅ SUCCESS — Final QA setup ready.


In [25]:
# =========================================================
# FINAL FILE INVENTORY
# =========================================================

expected_files = {
    "email_daily": EMAIL_DAILY,
    "usb_daily": USB_DAILY,
    "http_daily": HTTP_DAILY,
    "logon_daily": LOGON_DAILY,
    "user_day": USER_DAY,
    "benign_baseline": BASELINE,
    "behavior_features": BEHAVIOR,
    "ground_truth_users": GT_USERS,
    "ground_truth_events": GT_EVENTS,
    "malicious_pre_attack_baseline": MALICIOUS_BASELINE
}

inventory_rows = []

for name, path in expected_files.items():

    exists = path.exists()

    inventory_rows.append({
        "dataset": name,
        "exists": exists,
        "path": str(path)
    })

inventory = pd.DataFrame(inventory_rows)

print("=" * 110)
print("FINAL DATA ENGINEERING FILE INVENTORY")
print("=" * 110)

print(
    inventory.to_string(index=False)
)

assert inventory["exists"].all()

print(
    "\n✅ SUCCESS — All required Data Engineering files exist."
)

FINAL DATA ENGINEERING FILE INVENTORY
                      dataset  exists                                                                                                   path
                  email_daily    True                  C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\email_daily_r62.parquet
                    usb_daily    True                    C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\usb_daily_r62.parquet
                   http_daily    True                   C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\http_daily_r62.parquet
                  logon_daily    True                  C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\logon_daily_r62.parquet
                     user_day    True                  C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\user_day_r62_v1.parquet
              benign_baseline    True              C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\features\user

In [27]:
# =========================================================
# BUILD ATTACK-HOST ELIGIBILITY TABLE
# =========================================================

user_day_path = str(USER_DAY).replace("\\", "/")
baseline_path = str(BASELINE).replace("\\", "/")
attack_hosts_path = str(ATTACK_HOSTS).replace("\\", "/")

con = duckdb.connect()

con.execute(f"""
COPY (

WITH future_workdays AS (

    SELECT
        user,
        COUNT(*) AS future_workdays

    FROM read_parquet(
        '{user_day_path}'
    )

    WHERE day >= DATE '{CUTOFF}'
      AND is_weekend = 0
      AND logon_count > 0

    GROUP BY user
)

SELECT

    b.user,

    b.baseline_workdays,

    b.email_active_days,
    b.usb_active_days,
    b.http_upload_active_days,

    COALESCE(
        f.future_workdays,
        0
    ) AS future_workdays,


    -- =====================================================
    -- ELIGIBILITY LEVELS
    -- =====================================================

    CASE
        WHEN b.baseline_workdays >= 60
         AND b.email_active_days >= 30
         AND b.usb_active_days >= 30
         AND b.http_upload_active_days >= 30
         AND COALESCE(f.future_workdays, 0) >= 30
        THEN 1 ELSE 0
    END AS eligible_30,


    CASE
        WHEN b.baseline_workdays >= 60
         AND b.email_active_days >= 40
         AND b.usb_active_days >= 40
         AND b.http_upload_active_days >= 40
         AND COALESCE(f.future_workdays, 0) >= 30
        THEN 1 ELSE 0
    END AS eligible_40,


    CASE
        WHEN b.baseline_workdays >= 60
         AND b.email_active_days >= 50
         AND b.usb_active_days >= 50
         AND b.http_upload_active_days >= 50
         AND COALESCE(f.future_workdays, 0) >= 30
        THEN 1 ELSE 0
    END AS eligible_strict_50


FROM read_parquet(
    '{baseline_path}'
) b

LEFT JOIN future_workdays f
    ON b.user = f.user

ORDER BY b.user

)

TO '{attack_hosts_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
""")

con.close()

assert ATTACK_HOSTS.exists()

print("Created:")
print(ATTACK_HOSTS)

print(
    "\n✅ SUCCESS — Attack-host eligibility table created."
)

Created:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\features\attack_host_eligibility_r62.parquet

✅ SUCCESS — Attack-host eligibility table created.


In [29]:
# =========================================================
# ATTACK-HOST ELIGIBILITY VALIDATION
# =========================================================

con = duckdb.connect()

eligibility = con.execute(f"""
SELECT

    COUNT(*) AS benign_users,

    SUM(eligible_30)
        AS eligible_30,

    SUM(eligible_40)
        AS eligible_40,

    SUM(eligible_strict_50)
        AS eligible_strict_50

FROM read_parquet(
    '{attack_hosts_path}'
)
""").df()


bad_strict = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{attack_hosts_path}'
)

WHERE eligible_strict_50 = 1

AND (
       baseline_workdays < 60
    OR email_active_days < 50
    OR usb_active_days < 50
    OR http_upload_active_days < 50
    OR future_workdays < 30
)
""").fetchone()[0]

con.close()


print("=" * 100)
print("FINAL ATTACK-HOST ELIGIBILITY")
print("=" * 100)

print(
    eligibility.to_string(
        index=False
    )
)

print(
    "\nInvalid strict candidates:",
    bad_strict
)

assert int(
    eligibility.loc[0, "benign_users"]
) == 3995

assert int(
    eligibility.loc[0, "eligible_30"]
) == 199

assert int(
    eligibility.loc[0, "eligible_40"]
) == 158

assert int(
    eligibility.loc[0, "eligible_strict_50"]
) == 126

assert bad_strict == 0

print(
    "\n✅ SUCCESS — Final attack-host eligibility is correct."
)

FINAL ATTACK-HOST ELIGIBILITY
 benign_users  eligible_30  eligible_40  eligible_strict_50
         3995        199.0        158.0               126.0

Invalid strict candidates: 0

✅ SUCCESS — Final attack-host eligibility is correct.


In [31]:
# =========================================================
# POPULATION CONSISTENCY
# =========================================================

user_day_path = str(USER_DAY).replace("\\", "/")
baseline_path = str(BASELINE).replace("\\", "/")
gt_users_path = str(GT_USERS).replace("\\", "/")

con = duckdb.connect()

population = con.execute(f"""
SELECT

    (
        SELECT COUNT(DISTINCT user)
        FROM read_parquet(
            '{user_day_path}'
        )
    ) AS total_users,

    (
        SELECT COUNT(DISTINCT user)
        FROM read_parquet(
            '{baseline_path}'
        )
    ) AS benign_users,

    (
        SELECT COUNT(DISTINCT user)
        FROM read_parquet(
            '{gt_users_path}'
        )
    ) AS malicious_users
""").df()


overlap = con.execute(f"""
SELECT COUNT(*)

FROM (

    SELECT DISTINCT b.user

    FROM read_parquet(
        '{baseline_path}'
    ) b

    INNER JOIN read_parquet(
        '{gt_users_path}'
    ) g
        ON b.user = g.user
)
""").fetchone()[0]


union_users = con.execute(f"""
SELECT COUNT(DISTINCT user)

FROM (

    SELECT user
    FROM read_parquet(
        '{baseline_path}'
    )

    UNION ALL

    SELECT user
    FROM read_parquet(
        '{gt_users_path}'
    )
)
""").fetchone()[0]


con.close()


print("=" * 100)
print("POPULATION CONSISTENCY")
print("=" * 100)

print(
    population.to_string(
        index=False
    )
)

print("\nBenign/Malicious overlap:", overlap)
print("Combined unique users   :", union_users)


assert int(
    population.loc[0, "total_users"]
) == 4000

assert int(
    population.loc[0, "benign_users"]
) == 3995

assert int(
    population.loc[0, "malicious_users"]
) == 5

assert overlap == 0
assert union_users == 4000


print(
    "\n✅ SUCCESS — User populations are perfectly separated."
)

POPULATION CONSISTENCY
 total_users  benign_users  malicious_users
        4000          3995                5

Benign/Malicious overlap: 0
Combined unique users   : 4000

✅ SUCCESS — User populations are perfectly separated.


In [33]:
# =========================================================
# BEHAVIOR / BASELINE CONSISTENCY
# =========================================================

behavior_path = str(BEHAVIOR).replace("\\", "/")

con = duckdb.connect()

behavior_users = con.execute(f"""
SELECT COUNT(DISTINCT user)

FROM read_parquet(
    '{behavior_path}'
)
""").fetchone()[0]


missing_behavior_baseline = con.execute(f"""
SELECT COUNT(*)

FROM (

    SELECT DISTINCT f.user

    FROM read_parquet(
        '{behavior_path}'
    ) f

    LEFT JOIN read_parquet(
        '{baseline_path}'
    ) b
        ON f.user = b.user

    WHERE b.user IS NULL
)
""").fetchone()[0]


expected_future_users = con.execute(f"""
SELECT COUNT(DISTINCT d.user)

FROM read_parquet(
    '{user_day_path}'
) d

INNER JOIN read_parquet(
    '{baseline_path}'
) b
    ON d.user = b.user

WHERE d.day >= DATE '{CUTOFF}'
  AND d.is_weekend = 0
  AND d.logon_count > 0
""").fetchone()[0]


con.close()


print("Behavior users       :", behavior_users)
print("Expected future users:", expected_future_users)
print("Missing baselines    :", missing_behavior_baseline)


assert behavior_users == expected_future_users
assert missing_behavior_baseline == 0

print(
    "\n✅ SUCCESS — Behavior features match the benign future population."
)

Behavior users       : 3737
Expected future users: 3737
Missing baselines    : 0

✅ SUCCESS — Behavior features match the benign future population.


In [35]:
# =========================================================
# FINAL GROUND-TRUTH CONSISTENCY
# =========================================================

gt_events_path = str(GT_EVENTS).replace("\\", "/")
mal_baseline_path = str(MALICIOUS_BASELINE).replace("\\", "/")

con = duckdb.connect()

gt_check = con.execute(f"""
SELECT

    COUNT(*) AS answer_events,

    SUM(
        is_primary_actor_event
    ) AS primary_actor_events,

    SUM(
        is_context_event
    ) AS context_events,

    COUNT(
        DISTINCT malicious_actor
    ) AS malicious_actors

FROM read_parquet(
    '{gt_events_path}'
)
""").df()


malicious_baseline_users = con.execute(f"""
SELECT COUNT(DISTINCT user)

FROM read_parquet(
    '{mal_baseline_path}'
)
""").fetchone()[0]


missing_malicious_baseline = con.execute(f"""
SELECT COUNT(*)

FROM read_parquet(
    '{gt_users_path}'
) g

LEFT JOIN read_parquet(
    '{mal_baseline_path}'
) b
    ON g.user = b.user

WHERE b.user IS NULL
""").fetchone()[0]


con.close()


print("=" * 100)
print("GROUND TRUTH FINAL CONSISTENCY")
print("=" * 100)

print(
    gt_check.to_string(
        index=False
    )
)

print(
    "\nMalicious users with pre-attack baselines:",
    malicious_baseline_users
)

print(
    "Missing malicious baselines:",
    missing_malicious_baseline
)


assert int(
    gt_check.loc[0, "answer_events"]
) == 470

assert int(
    gt_check.loc[0, "primary_actor_events"]
) == 420

assert int(
    gt_check.loc[0, "context_events"]
) == 50

assert int(
    gt_check.loc[0, "malicious_actors"]
) == 5

assert malicious_baseline_users == 5
assert missing_malicious_baseline == 0


print(
    "\n✅ SUCCESS — Official malicious ground truth is fully consistent."
)

GROUND TRUTH FINAL CONSISTENCY
 answer_events  primary_actor_events  context_events  malicious_actors
           470                 420.0            50.0                 5

Malicious users with pre-attack baselines: 5
Missing malicious baselines: 0

✅ SUCCESS — Official malicious ground truth is fully consistent.


In [37]:
# =========================================================
# GENERATE FINAL FILE SUMMARY
# =========================================================

files_for_summary = {
    "email_daily_r62": EMAIL_DAILY,
    "usb_daily_r62": USB_DAILY,
    "http_daily_r62": HTTP_DAILY,
    "logon_daily_r62": LOGON_DAILY,
    "user_day_r62_v1": USER_DAY,
    "user_baseline_r62_v1": BASELINE,
    "behavior_features_r62_v1": BEHAVIOR,
    "attack_host_eligibility_r62": ATTACK_HOSTS,
    "ground_truth_users_r62": GT_USERS,
    "ground_truth_events_r62": GT_EVENTS,
    "malicious_pre_attack_baseline_r62": MALICIOUS_BASELINE
}

summary_rows = []

con = duckdb.connect()

for name, path in files_for_summary.items():

    p = str(path).replace("\\", "/")

    row_count = con.execute(f"""
        SELECT COUNT(*)
        FROM read_parquet('{p}')
    """).fetchone()[0]

    schema = con.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet('{p}')
    """).df()

    summary_rows.append({
        "dataset": name,
        "rows": row_count,
        "columns": len(schema),
        "file_path": str(path)
    })

con.close()

file_summary = pd.DataFrame(
    summary_rows
)

summary_output = (
    RESULTS /
    "dataset_inventory_r62.csv"
)

file_summary.to_csv(
    summary_output,
    index=False
)

print("=" * 110)
print("FINAL DATASET SUMMARY")
print("=" * 110)

print(
    file_summary.to_string(
        index=False
    )
)

print("\nSaved:")
print(summary_output)

print(
    "\n✅ SUCCESS — Final file summary exported."
)

FINAL DATASET SUMMARY
                          dataset    rows  columns                                                                                              file_path
                  email_daily_r62 1381557       13                  C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\email_daily_r62.parquet
                    usb_daily_r62  308647       11                    C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\usb_daily_r62.parquet
                   http_daily_r62 1391239        8                   C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\http_daily_r62.parquet
                  logon_daily_r62 1394010        6                  C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\logon_daily_r62.parquet
                  user_day_r62_v1 1394010       34                  C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\processed\user_day_r62_v1.parquet
             user_baseline_r62_v1    3995       36    

In [39]:
# =========================================================
# GENERATE DATA DICTIONARY
# =========================================================

dictionary_rows = []

con = duckdb.connect()

for dataset_name, path in files_for_summary.items():

    p = str(path).replace("\\", "/")

    schema = con.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet('{p}')
    """).df()

    for _, row in schema.iterrows():

        dictionary_rows.append({
            "dataset":
                dataset_name,

            "column_name":
                row["column_name"],

            "data_type":
                row["column_type"],

            "nullable":
                row["null"]
        })

con.close()


data_dictionary = pd.DataFrame(
    dictionary_rows
)


dictionary_output = (
    RESULTS /
    "schema_dictionary_r62.csv"
)

data_dictionary.to_csv(
    dictionary_output,
    index=False
)


print(
    "Dictionary rows:",
    len(data_dictionary)
)

print("Saved:")
print(dictionary_output)

print(
    "\n✅ SUCCESS — Data dictionary exported."
)

Dictionary rows: 273
Saved:
C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\results\schema_dictionary_r62.csv

✅ SUCCESS — Data dictionary exported.


In [41]:
# =========================================================
# FINAL DATA ENGINEERING COMPLETION REPORT
# =========================================================

print("=" * 100)
print("CERT r6.2 — DATA ENGINEERING FINAL STATUS")
print("=" * 100)

print("""
[01] Dataset feasibility audit                  ✅ COMPLETE
[02] Email daily aggregation                   ✅ COMPLETE
[03] USB/File daily aggregation                ✅ COMPLETE
[04] HTTP daily aggregation                    ✅ COMPLETE
[05] Logon daily aggregation                   ✅ COMPLETE
[06] Unified User-Day dataset                  ✅ COMPLETE
[07] Temporal cutoff selection                 ✅ COMPLETE
[08] Benign user baseline profiles             ✅ COMPLETE
[09] Baseline-relative behavioral features     ✅ COMPLETE
[10] Official user-level ground truth          ✅ COMPLETE
[11] Official event-level ground truth         ✅ COMPLETE
[12] Primary actor / context separation        ✅ COMPLETE
[13] Malicious pre-attack baselines            ✅ COMPLETE
[14] Attack-host eligibility metadata          ✅ COMPLETE
[15] Cross-file consistency validation         ✅ COMPLETE
[16] Final file inventory                      ✅ COMPLETE
[17] Technical data dictionary                 ✅ COMPLETE
""")

print("-" * 100)

print("Official cutoff:")
print("2011-01-01")

print("\nMain baseline population:")
print("3995 benign users")

print("\nOriginal CERT malicious population:")
print("5 users")

print("\nStrict synthetic attack-host candidates:")
print("126 benign users")

print("\n" + "=" * 100)
print("✅ DATA ENGINEERING R6.2 — COMPLETE")
print("=" * 100)

CERT r6.2 — DATA ENGINEERING FINAL STATUS

[01] Dataset feasibility audit                  ✅ COMPLETE
[02] Email daily aggregation                   ✅ COMPLETE
[03] USB/File daily aggregation                ✅ COMPLETE
[04] HTTP daily aggregation                    ✅ COMPLETE
[05] Logon daily aggregation                   ✅ COMPLETE
[06] Unified User-Day dataset                  ✅ COMPLETE
[07] Temporal cutoff selection                 ✅ COMPLETE
[08] Benign user baseline profiles             ✅ COMPLETE
[09] Baseline-relative behavioral features     ✅ COMPLETE
[10] Official user-level ground truth          ✅ COMPLETE
[11] Official event-level ground truth         ✅ COMPLETE
[12] Primary actor / context separation        ✅ COMPLETE
[13] Malicious pre-attack baselines            ✅ COMPLETE
[14] Attack-host eligibility metadata          ✅ COMPLETE
[15] Cross-file consistency validation         ✅ COMPLETE
[16] Final file inventory                      ✅ COMPLETE
[17] Technical data dictiona

In [43]:
# =========================================================
# SEMANTIC FEATURE DICTIONARY VALIDATION
# =========================================================

FEATURE_DICTIONARY = (
    RESULTS /
    "feature_dictionary_r62.csv"
)

assert FEATURE_DICTIONARY.exists()

feature_dictionary = pd.read_csv(
    FEATURE_DICTIONARY
)

assert len(feature_dictionary) == 85

assert (
    feature_dictionary[
        "feature_name"
    ].nunique()
    == 85
)

print(
    "Feature dictionary rows:",
    len(feature_dictionary)
)

print(
    "\n✅ SUCCESS — Semantic Feature Dictionary is complete."
)


Feature dictionary rows: 85

✅ SUCCESS — Semantic Feature Dictionary is complete.
